In [25]:
import duckdb
import io
from ollama import chat
import pandas as pd
import numpy as np

In [26]:
rawdata = './data/HomeIntellex_1.csv'
outpath = './data/HomeIntellex_1.parquet'

colnames = ['status', 'date', 'time', 'sensor']
df = pd.read_csv(rawdata, header=None, names=colnames)

df.to_parquet(outpath)

In [27]:
model = 'qwen3:8b'

def get_activities(data, step = 1, previous = []):
    print(f'Summarizing step {step}...')
    format = '|Activity|Start Time|End Time|Duration|Notes|'
    system_prompt = f"You are a data scientist tasked with interpreting home sensor data from sensors placed around a subject's house. Provide your answers in the following tabular format for easy parsing: {format}"

    if previous:
        context = "Use the last timestep's analysis for context {previous}"

    # Chat with a system prompt
    response = chat(model, 
                    messages=[
                        {'role': 'system', 'content': system_prompt},
                        {'role': 'user', 'content': f'Create a summary of activities within the following csv time window. Include in your summary your best guess as to what might be happening: {data}'}
                    ])
    # We want to grab the table from the output
    table_str = "\n".join([line for line in response.message.content.split('\n') if line.strip().startswith('|')])

    # Read into a DataFrame
    df = pd.read_csv(io.StringIO(table_str), sep="|", skipinitialspace=True).dropna(axis=1, how='all')

    # Clean column names (removing whitespace)
    df.columns = df.columns.str.strip()

    # Remove '---' from rows (part of the way the llm generates tables)
    df = df.replace(r'-{2,}', np.nan, regex=True)
    df = df.dropna()

    print(df)
    # Return the activities summary
    return(df.to_dict('records'))



In [28]:
data_location = './data/HomeIntellex_1.parquet'

db = duckdb.connect()
db.execute(f"CREATE VIEW subject1 AS SELECT * FROM read_parquet('{data_location}')")

# Set timestep boundaries
start = 0
stop = 500
step = 250

# Initialize the activities dictionary
activities = []
previous=[]

query = "SELECT * FROM subject1 LIMIT ? OFFSET ?"

for offset in range(start, stop, step):
    timestep = db.execute(query, [step, offset]).df()
    index = round(offset / step) + 1
    response = get_activities(timestep, index, previous)
    activities.extend(response)
    previous = response

db.close()

Summarizing step 1...
                 Activity Start Time  End Time  Duration  \
1          Entering House   20:00:03  20:03:02  00:02:59   
2           In Livingroom   20:03:02  20:48:18  00:45:16   
3              In Kitchen   20:48:18  20:48:21  00:00:03   
4  Re-entering Livingroom   20:48:21  20:49:08  00:00:47   
5           Leaving House   20:03:02  20:00:03   Invalid   

                                               Notes  
1  Multiple "open" events at entrance sensor sugg...  
2  Presence detected in living room until exit ev...  
3  Short duration of motion detection in kitchen,...  
4  Subject returns to living room after leaving, ...  
5  No clear exit event detected; sensor data sugg...  
Summarizing step 2...
                             Activity Start Time  End Time  \
1  Moving from Living Room to Bedroom   20:49:10  20:49:13   
2                     Leaving Bedroom   20:49:13  20:49:18   
3                           In Office   22:23:41  22:25:49   
4         Final E

In [29]:
# Export output to csv
df = pd.DataFrame.from_dict(activities)
print(df)
df.to_csv('./data/output.csv')

                             Activity Start Time  End Time  \
0                      Entering House   20:00:03  20:03:02   
1                       In Livingroom   20:03:02  20:48:18   
2                          In Kitchen   20:48:18  20:48:21   
3              Re-entering Livingroom   20:48:21  20:49:08   
4                       Leaving House   20:03:02  20:00:03   
5  Moving from Living Room to Bedroom   20:49:10  20:49:13   
6                     Leaving Bedroom   20:49:13  20:49:18   
7                           In Office   22:23:41  22:25:49   
8         Final Exit from Living Room   20:49:18  20:49:18   

              Duration                                              Notes  
0             00:02:59  Multiple "open" events at entrance sensor sugg...  
1             00:45:16  Presence detected in living room until exit ev...  
2             00:00:03  Short duration of motion detection in kitchen,...  
3             00:00:47  Subject returns to living room after leaving, ...  